In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
list_files = os.listdir("../")
list_files

['away_team.csv',
 'away_team_score.csv',
 'event.csv',
 'home_team.csv',
 'home_team_score.csv',
 'Javadi',
 'notebook.ipynb',
 'odds.csv',
 'pbp.csv',
 'power.csv',
 'round.csv',
 'season.csv',
 'statistics.csv',
 'Tennis-project.zip',
 'time.csv',
 'tournament.csv',
 'venue.csv',
 'votes.csv']

In [3]:
event_df = pd.read_csv("../event.csv")
home_team_df = pd.read_csv("../home_team.csv")
away_team_df = pd.read_csv("../away_team.csv")
round_info_df = pd.read_csv("../round.csv")
tournament_df = pd.read_csv("../tournament.csv")


In [4]:

# ─────────────────────────────────────────────────────────────────────────────
# THRESHOLDS
# Open Era start: 1968-01-01 → Unix timestamp 0 is 1970, so min is -63158400
# We'll use a safe floor of Jan 1 2000 for this dataset (modern era)
# Max realistic tournament wins in one month: 3
# ─────────────────────────────────────────────────────────────────────────────
MAX_WINS_PER_MONTH = 3

# Final round name variants found across different data sources
FINAL_ROUND_NAMES = {
    'final', 'f', 'finals', 'championship', 'championship match',
    'gold medal match', 'the final', 'grand final'
}

report         = []
total_original = len(event_df)

def log(step, desc, removed, note=''):
    report.append({
        'Step'          : step,
        'Description'   : desc,
        'Rows Removed'  : removed,
        '% of Original' : round(removed / total_original * 100, 2),
        'Note'          : note
    })

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Filter event_df — only finished matches with valid winner_code
# ─────────────────────────────────────────────────────────────────────────────
df_event = event_df.copy()
before   = len(df_event)
df_event = df_event[df_event['winner_code'].isin([1, 2])].copy()
removed  = before - len(df_event)
log(1, 'Invalid or null winner_code (unfinished/corrupt matches)', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Validate and convert start_datetime (Unix timestamp → datetime)
# ─────────────────────────────────────────────────────────────────────────────
before = len(df_event)

# Drop null timestamps
df_event = df_event[df_event['start_datetime'].notna()].copy()

# Drop zero timestamps
df_event = df_event[df_event['start_datetime'] != 0].copy()

# Convert to datetime
df_event['match_date'] = pd.to_datetime(
    df_event['start_datetime'], unit='s', utc=True
).dt.tz_localize(None)

df_event['year_month'] = df_event['match_date'].dt.to_period('M')
df_event['month_label'] = df_event['match_date'].dt.strftime('%B %Y')

removed = before - len(df_event)
log(2, 'Null/zero start_datetime timestamps', removed, '')

valid_match_ids = set(df_event['match_id'])

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Filter MatchRoundInfo to FINALS only
# ─────────────────────────────────────────────────────────────────────────────
round_df = round_info_df.copy()

# Normalise round names for matching
round_df['name_norm'] = round_df['name'].str.strip().str.lower()

# Keep only final rounds
finals_df = round_df[round_df['name_norm'].isin(FINAL_ROUND_NAMES)].copy()

# Only keep match_ids that survived event cleaning
finals_df = finals_df[finals_df['match_id'].isin(valid_match_ids)].copy()

print(f"  Round name variants found for finals:")
print(f"  {finals_df['name'].value_counts().to_dict()}")
print(f"  Total final matches identified: {len(finals_df):,}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Drop duplicate match_ids in finals table
# A match cannot be both a final and something else simultaneously
# ─────────────────────────────────────────────────────────────────────────────
before       = len(finals_df)
finals_df    = finals_df.drop_duplicates(subset='match_id')
removed      = before - len(finals_df)
log(4, 'Duplicate match_ids in round table', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Filter tournament table — singles only, valid tournament_id
# competition_type: filter to singles (check what values exist first)
# ─────────────────────────────────────────────────────────────────────────────
tourn_df = tournament_df.copy()

# Drop null tournament_ids
before    = len(tourn_df)
tourn_df  = tourn_df[tourn_df['tournament_id'].notna()].copy()

# Drop duplicate match_ids in tournament table
tourn_df  = tourn_df.drop_duplicates(subset='match_id')

# Filter to valid match_ids
tourn_df  = tourn_df[tourn_df['match_id'].isin(valid_match_ids)].copy()

# Log competition_type distribution before filtering
print(f"\n  Competition type distribution:")
print(f"  {tourn_df['competition_type'].value_counts().to_dict()}")

# Filter singles — competition_type = 1 is typically singles
# Keep only if we can confirm; otherwise keep all and flag
if tourn_df['competition_type'].notna().any():
    singles_types = tourn_df['competition_type'].value_counts()
    # Keep the most common type(s) — singles is the dominant competition
    # in this dataset; adjust the value if your data uses a different code
    dominant_type = singles_types.index[0]
    tourn_singles = tourn_df[tourn_df['competition_type'] == dominant_type].copy()
    print(f"\n  Keeping competition_type={dominant_type} as singles "
          f"({len(tourn_singles):,} matches)")
else:
    tourn_singles = tourn_df.copy()
    print("\n  competition_type all null — keeping all tournaments")

log(5, 'Non-singles competition types and invalid tournament_ids',
    total_original - len(tourn_singles), 'Keeps only dominant competition type')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: Merge finals → event → tournament to get one row per final
# ─────────────────────────────────────────────────────────────────────────────
# Start with finals match_ids
df = finals_df[['match_id']].copy()

# Merge event info (winner_code, date)
df = df.merge(
    df_event[['match_id', 'winner_code', 'match_date',
              'year_month', 'month_label']],
    on='match_id', how='inner'
)

# Merge tournament info
df = df.merge(
    tourn_singles[['match_id', 'tournament_id', 'tournament_name']],
    on='match_id', how='inner'
)

print(f"\n  Finals after merging all tables: {len(df):,}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 7 (REVISED): One final per tournament — distinguish corrupt duplicates
# from genuinely interrupted finals
# ─────────────────────────────────────────────────────────────────────────────
before             = len(df)
resolved_finals    = []
corrupt_tourn_ids  = []
interrupted_n      = 0

for tourn_id, group in df.groupby('tournament_id'):

    # Only one row — no issue at all
    if len(group) == 1:
        resolved_finals.append(group.iloc[0])
        continue

    # ── Case A: Same match_id appearing multiple times ────────────────────
    # This is always corrupt — one match_id cannot be two different events
    if group['match_id'].nunique() == 1:
        # All rows refer to the same match — take first, drop the rest
        resolved_finals.append(group.iloc[0])
        continue

    # ── Case B: Different match_ids for the same tournament final ─────────
    # Now we apply interruption logic to distinguish corrupt vs interrupted

    # Test 1: Do the winner_codes agree across all rows?
    # A real interrupted match that was completed should always have the
    # same winner. Different winner_codes = corrupt data
    if group['winner_code'].nunique() > 1:
        corrupt_tourn_ids.append(tourn_id)
        continue

    # Test 2: Are the match dates meaningfully different?
    # An interrupted match resumes within a few days (typically 1–3 days).
    # Rows more than 7 days apart = likely two different finals = corrupt
    dates     = pd.to_datetime(group['match_date'])
    date_diff = (dates.max() - dates.min()).days
    if date_diff > 7:
        corrupt_tourn_ids.append(tourn_id)
        continue

    # Test 3: Do the home/away players agree across rows?
    # An interrupted match must have the same two players.
    # Different player combinations = corrupt (two different finals)
    home_players_in_group = group['home_player_id'].nunique()
    away_players_in_group = group['away_player_id'].nunique()
    if home_players_in_group > 1 or away_players_in_group > 1:
        corrupt_tourn_ids.append(tourn_id)
        continue

    # Test 4: Are the rows nearly identical in all fields?
    # If winner_code, players, and date are the same — it's a pure
    # duplicate, not a real interruption (just bad data entry)
    check_cols    = ['winner_code', 'home_player_id', 'away_player_id']
    available     = [c for c in check_cols if c in group.columns]
    is_identical  = (group[available].nunique() == 1).all()
    date_too_close = date_diff == 0
    if is_identical and date_too_close:
        # Pure duplicate — keep one row
        resolved_finals.append(group.iloc[0])
        continue

    # ── Passes all tests: treat as genuinely interrupted final ────────────
    # Keep the LATEST row (final session = when match was completed)
    # and note it as interrupted
    resolved_finals.append(
        group.sort_values('match_date').iloc[-1]
    )
    interrupted_n += 1

# Rebuild df from resolved finals
df_resolved = pd.DataFrame(resolved_finals)

# Remove corrupt tournament finals entirely
df_resolved = df_resolved[
    ~df_resolved['tournament_id'].isin(corrupt_tourn_ids)
].copy()

removed = before - len(df_resolved)
df      = df_resolved.reset_index(drop=True)

log(7,
    f'Duplicate finals per tournament_id — '
    f'{interrupted_n} interrupted (merged), '
    f'{len(corrupt_tourn_ids)} corrupt (removed)',
    removed,
    'Interrupted: same players + same winner + dates within 7 days')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 8: Load and clean player tables
# ─────────────────────────────────────────────────────────────────────────────
def clean_player_table(df_player, label):
    out = df_player[
        ['match_id', 'player_id', 'name', 'full_name', 'country', 'gender']
    ].copy()
    out = out.drop_duplicates(subset='match_id')
    out = out[out['match_id'].isin(set(df['match_id']))]
    out = out[out['player_id'].notna()].copy()
    print(f"  [{label}] {len(out):,} rows after cleaning")
    return out

home_players = clean_player_table(home_team_df, 'HOME')
away_players = clean_player_table(away_team_df, 'AWAY')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 9: Identify winner player_id, name, country, gender per final
# winner_code=1 → home player won; winner_code=2 → away player won
# ─────────────────────────────────────────────────────────────────────────────
df = df.merge(
    home_players[['match_id', 'player_id', 'name', 'full_name',
                  'country', 'gender']]
    .rename(columns={
        'player_id' : 'home_player_id',
        'name'      : 'home_name',
        'full_name' : 'home_fullname',
        'country'   : 'home_country',
        'gender'    : 'home_gender'
    }),
    on='match_id', how='left'
)
df = df.merge(
    away_players[['match_id', 'player_id', 'name', 'full_name',
                  'country', 'gender']]
    .rename(columns={
        'player_id' : 'away_player_id',
        'name'      : 'away_name',
        'full_name' : 'away_fullname',
        'country'   : 'away_country',
        'gender'    : 'away_gender'
    }),
    on='match_id', how='left'
)

# Assign winner attributes
df['winner_player_id'] = np.where(
    df['winner_code'] == 1,
    df['home_player_id'], df['away_player_id']
)
df['winner_name'] = np.where(
    df['winner_code'] == 1,
    df['home_fullname'].fillna(df['home_name']),
    df['away_fullname'].fillna(df['away_name'])
)
df['winner_country'] = np.where(
    df['winner_code'] == 1,
    df['home_country'], df['away_country']
)
df['winner_gender'] = np.where(
    df['winner_code'] == 1,
    df['home_gender'], df['away_gender']
)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 10: Drop rows with null winner_player_id or null winner_name
# ─────────────────────────────────────────────────────────────────────────────
before  = len(df)
df      = df[df['winner_player_id'].notna() & df['winner_name'].notna()].copy()
removed = before - len(df)
log(10, 'Finals where winner player_id or name is null', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 11: Validate gender — standardise and remove unknowns
# ─────────────────────────────────────────────────────────────────────────────
before = len(df)
df['winner_gender'] = df['winner_gender'].str.strip().str.upper()
df['winner_gender'] = df['winner_gender'].map(
    lambda x: 'M' if x in ['M', 'MALE', 'MEN', '1', 'ATP']
    else ('F' if x in ['F', 'FEMALE', 'WOMEN', '2', 'WTA']
    else np.nan)
)
df     = df[df['winner_gender'].notna()].copy()
removed = before - len(df)
log(11, 'Matches with unrecognised or null gender label', removed, '')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 12: Resolve player name — use most frequent name per player_id
# (handles minor spelling inconsistencies across matches)
# ─────────────────────────────────────────────────────────────────────────────
canonical_name = (
    df.groupby('winner_player_id')['winner_name']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
    .rename(columns={'winner_name': 'canonical_name'})
)
df = df.merge(canonical_name, on='winner_player_id', how='left')

# ─────────────────────────────────────────────────────────────────────────────
# STEP 13: Count tournament wins per player per month
# ─────────────────────────────────────────────────────────────────────────────
monthly_wins = (
    df.groupby(['winner_player_id', 'canonical_name',
                'winner_country', 'winner_gender', 'year_month', 'month_label'])
    .agg(
        tournaments_won = ('tournament_id',   'nunique'),
        tournament_names = ('tournament_name', lambda x: ', '.join(x.unique()))
    )
    .reset_index()
    .sort_values('tournaments_won', ascending=False)
)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 14: Flag and remove physically impossible win counts
# Max = 3 per month (one tournament per week, ~4 weeks per month,
# but players need rest/travel between tournaments)
# ─────────────────────────────────────────────────────────────────────────────
before      = len(monthly_wins)
suspicious  = monthly_wins[monthly_wins['tournaments_won'] > MAX_WINS_PER_MONTH]
if len(suspicious) > 0:
    print(f"\n  ⚠ Suspicious entries (>{MAX_WINS_PER_MONTH} wins/month):")
    print(suspicious[['canonical_name', 'month_label',
                       'tournaments_won', 'tournament_names']].to_string(index=False))
monthly_wins = monthly_wins[
    monthly_wins['tournaments_won'] <= MAX_WINS_PER_MONTH
].copy()
removed      = before - len(monthly_wins)
log(14, f'Entries with >{MAX_WINS_PER_MONTH} tournament wins in one month (impossible)',
    removed, 'Physical ceiling: ~1 tournament per week')

# ─────────────────────────────────────────────────────────────────────────────
# PRINT CLEANING REPORT
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 75)
print("DATA CLEANING REPORT — Most Tournament Wins in a Single Month")
print("=" * 75)
print(f"  Original event rows        : {total_original:,}")
print(f"  Finals identified          : {len(df):,}")
print(f"  Player-month combinations  : {len(monthly_wins):,}")
print("=" * 75)
print(pd.DataFrame(report).to_string(index=False))





  Round name variants found for finals:
  {'Final': 533}
  Total final matches identified: 533

  Competition type distribution:
  {2.0: 16100, 1.0: 12}

  Keeping competition_type=2.0 as singles (16,100 matches)

  Finals after merging all tables: 499
  [HOME] 250 rows after cleaning
  [AWAY] 253 rows after cleaning

DATA CLEANING REPORT — Most Tournament Wins in a Single Month
  Original event rows        : 35,053
  Finals identified          : 252
  Player-month combinations  : 220
 Step                                                                      Description  Rows Removed  % of Original                                                          Note
    1                         Invalid or null winner_code (unfinished/corrupt matches)          3049           8.70                                                              
    2                                              Null/zero start_datetime timestamps             0           0.00                                       

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# RESULTS — split by gender
# ─────────────────────────────────────────────────────────────────────────────
for gender, label in [('M', 'ATP — MEN'), ('F', 'WTA — WOMEN')]:
    subset = monthly_wins[monthly_wins['winner_gender'] == gender].head(10)
    if len(subset) == 0:
        continue

    print(f"\n{'=' * 75}")
    print(f"TOP 10 — {label}")
    print(f"{'=' * 75}")
    print(f"  {'Rank':<5} {'Player':<25} {'Country':<10} "
          f"{'Month':<15} {'Wins':>5}  Tournaments")
    print(f"  {'-'*5} {'-'*25} {'-'*10} {'-'*15} {'-'*5}  {'-'*30}")

    for i, (_, row) in enumerate(subset.iterrows(), 1):
        print(f"  {i:<5} {row['canonical_name']:<25} "
              f"{row['winner_country']:<10} "
              f"{row['month_label']:<15} "
              f"{int(row['tournaments_won']):>5}  "
              f"{row['tournament_names']}")


TOP 10 — ATP — MEN
  Rank  Player                    Country    Month            Wins  Tournaments
  ----- ------------------------- ---------- --------------- -----  ------------------------------
  1     Popko, Dmitry             Kazakhstan February 2024       3  Sunrise, Singles Main, M-ITF-USA-03A, Palm Coast, Singles Main, M-ITF-USA-04A, Naples, Singles Main, M-ITF-USA-05A
  2     Jianu, Filip Cristian     Romania    March 2024          3  Kish Island, Singles Main, M-ITF-IRI-03A, Kish Island, Singles Main, M-ITF-IRI-04A, Antalya, Singles Main, M-ITF-TUR-11A
  3     Gengel, Marek             Czech Republic February 2024       3  Sharm ElSheikh, Singles Main, M-ITF-EGY-01A, Sharm ElSheikh, Singles Main, M-ITF-EGY-02A, Sharm ElSheikh, Singles Main, M-ITF-EGY-03A
  4     Nicod, Jakub              Czech Republic March 2024          3  Monastir, Singles Main, M-ITF-TUN-13A, Monastir, Singles Main, M-ITF-TUN-19A, Trnava, Singles Main, M-ITF-SVK-01A
  5     Faria, Jaime              Por

In [6]:
# Overall winner
print(f"\n{'=' * 75}")
print("OVERALL ANSWER")
print(f"{'=' * 75}")
top = monthly_wins.iloc[0]
print(f"\n  Player  : {top['canonical_name']}")
print(f"  Country : {top['winner_country']}")
print(f"  Month   : {top['month_label']}")
print(f"  Wins    : {int(top['tournaments_won'])}")
print(f"  Tourneys: {top['tournament_names']}")


OVERALL ANSWER

  Player  : Popko, Dmitry
  Country : Kazakhstan
  Month   : February 2024
  Wins    : 3
  Tourneys: Sunrise, Singles Main, M-ITF-USA-03A, Palm Coast, Singles Main, M-ITF-USA-04A, Naples, Singles Main, M-ITF-USA-05A
